# SPACESHIP TITANIC

In [7]:
import copy
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## Data Import

In [8]:
train_data = pd.read_csv("./data/train.csv")
test_data = pd.read_csv("./data/test.csv")

## Data Preprocess
- Transported/CryoSleep 轉成 0/1
- PassengerId、Name、Cabin 不直接當特徵
- Cabin 缺失值補上 Missing//Missing
    - 把 Cabin 拆成 CabinDeck、CabinNum、CabinSide


In [9]:
def to_binary(series):
    return series.astype(str).map({"False": 0, "True": 1}).astype(float)


def add_basic_features(df):
    df = df.copy()
    df = df.drop(columns=["PassengerId", "Name"], errors="ignore")
    cabin = df["Cabin"].fillna("Missing//Missing").str.split("/", expand=True)
    df["CabinDeck"] = cabin[0]
    df["CabinNum"] = pd.to_numeric(cabin[1], errors="coerce")
    df["CabinSide"] = cabin[2]
    return df.drop(columns=["Cabin"])


# Split train / validation first.
X = train_data.drop(columns=["Transported"]).copy()
y = to_binary(train_data["Transported"])
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train = add_basic_features(X_train)
X_val = add_basic_features(X_val)


## Baseline 0
先看一個最簡單的 `Transported = CryoSleep` validation accuracy
- 把 validation set 裡 CryoSleep 的缺失值，補成算出的眾數

In [ ]:
# 先從 training split 的 CryoSleep 找出最常出現的值
cryo_mode = X_train["CryoSleep"].mode(dropna=True)[0]
# 把 validation split 裡 CryoSleep 的缺失值，補成剛剛算出的眾數
cryo_pred = to_binary(X_val["CryoSleep"].fillna(cryo_mode))
# 直接算 accuracy
acc_b0 = accuracy_score(y_val, cryo_pred >= 0.5)
print(f"Baseline 0 | Transported = CryoSleep | Acc: {acc_b0:.4f}")

## Baseline1
Ridge fit training split，再算 validation accuracy

In [ ]:
# Baseline 1: Ridge on train split, evaluated on validation split.
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")), # 數值欄位用 median 補缺失值
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")), # 類別欄位用 most_frequent 補缺失值
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("ridge", Ridge(alpha=1.0)),
])

model.fit(X_train, y_train)
val_pred = model.predict(X_val)
acc_b1 = accuracy_score(y_val, val_pred >= 0.5)
print(f"Baseline 1 | Ridge val Acc: {acc_b1:.4f}")


# Pytorch